In [ ]:
# ==========================================
# 1. COLAB SETUP: DRIVE + REPO CODE
# ==========================================
import os
import pathlib
import subprocess
import sys

from google.colab import drive
from google.colab import userdata

REPO_URL = "https://github.com/DrHBSB/RadLE_CRASH_Lab.git"
REPO_DIR = pathlib.Path("/content/RadLE_CRASH_Lab")
SRC_DIR = REPO_DIR / "src"
MODULE_PATH = SRC_DIR / "radle_benchmark.py"
github_token = None

# Drive remains the home for images and benchmark outputs.
drive.mount("/content/drive")


def run_git(args, check=True):
    """Run git and print useful stderr without leaking the optional token."""
    result = subprocess.run(args, text=True, capture_output=True)
    stdout = result.stdout.replace(github_token, "***") if github_token else result.stdout
    stderr = result.stderr.replace(github_token, "***") if github_token else result.stderr
    if stdout.strip():
        print(stdout)
    if stderr.strip():
        print(stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Git command failed with exit code {result.returncode}: {args[:2]}")
    return result


# Colab does not automatically check out the full GitHub repo when opening a notebook.
# For this private repo, add a Colab secret named GITHUB_TOKEN only if unauthenticated clone fails.
if (REPO_DIR / ".git").exists():
    pull_result = run_git(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    if pull_result.returncode != 0:
        print("WARNING: git pull failed; continuing with the existing Colab checkout.")
elif not MODULE_PATH.exists():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git checkout and {MODULE_PATH} was not found. "
            "Restart the Colab runtime or remove that folder, then rerun this setup cell."
        )

    try:
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = None

    if not github_token:
        raise RuntimeError(
            "This private GitHub repo needs a Colab secret named GITHUB_TOKEN "
            "so the Colab runtime can clone src/radle_benchmark.py. "
            "Add the secret, restart or rerun this setup cell, then continue."
        )

    clone_url = REPO_URL.replace("https://", f"https://{github_token}@")
    run_git(["git", "clone", clone_url, str(REPO_DIR)])

if not MODULE_PATH.exists():
    raise RuntimeError(f"Benchmark module not found after setup: {MODULE_PATH}")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


In [ ]:
# ==========================================
# 2. API CLIENT + BENCHMARK IMPORTS
# ==========================================
from openai import OpenAI

from radle_benchmark import create_scorer_view, run_benchmark

try:
    os.environ["TEST_OPENROUTER_API_KEY"] = userdata.get("TEST_OPENROUTER_API_KEY")
except Exception:
    print("ERROR: Could not find TEST_OPENROUTER_API_KEY in Colab Secrets.")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["TEST_OPENROUTER_API_KEY"],
)


In [ ]:
# ==========================================
# 3. DRIVE PATHS + RUN CONFIG
# ==========================================
master_images_folder = "/content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/RadLE v2 Master Data"
final_output_csv = "/content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/RadLE_v1.5_RSNA.csv"

# Recommended sequence: run TEST_LIMIT=1 first; if syntax/API plumbing passes, run TEST_LIMIT=5.
# Use TEST_LIMIT=None only when ready for the full benchmark.
TEST_LIMIT = 1


In [ ]:
# ==========================================
# 4. RUN BENCHMARK
# ==========================================
df_final = run_benchmark(
    client=client,
    image_folder=master_images_folder,
    output_csv=final_output_csv,
    test_limit=TEST_LIMIT,
)

print("\nFINAL DATAFRAME PREVIEW:")
from IPython.display import display

display(df_final.head())


In [ ]:
# ==========================================
# 5. SCORER'S VIEW & PIVOT
# ==========================================
df_scorer, display_df, scorer_csv = create_scorer_view(final_output_csv)

print(f"Scorer version saved to: {scorer_csv}")
print("\nTRANSPOSED CONSENSUS VIEW:")
display(display_df)
